In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [5]:
# TODO: Read image data, feed into Claude
with open("prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode('utf-8')

messages = []

add_user_message(messages, [
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes
        }
    },
    {
        "type": "text",
        "text": prompt
    }
])

chat(messages)

Message(id='msg_012i11st4WyQv8CNMYU9Kiq1', container=None, content=[TextBlock(citations=None, text='# Satellite Image Analysis - Fire Risk Assessment\n\n**1. Residence Identification:**\nThe primary residence is a single-story structure with a light-colored (gray) roof located in the center of the image, featuring an irregular rectangular shape with what appears to be an attached garage or extension on the eastern side, surrounded by dense vegetation on all sides.\n\n**2. Tree Overhang Analysis:**\nMultiple trees have canopies that extend directly over portions of the residence roof, with overhanging branches covering approximately 40-50% of the total roof area, particularly dense on the northern, eastern, and southern sides of the structure.\n\n**3. Fire Risk Assessment:**\nThe overhanging trees create significant wildfire vulnerability by providing direct fuel pathways to the roof surface, with multiple ember catch points where branches and leaves could accumulate, and the dense cano

In [7]:
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode('utf-8')

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": file_bytes
            },
            "title": "earth.pdf",
            "citations": { "enabled": True}
        },
        {
            "type": "text",
            "text": "How were Earth's atmosphere and oceans were formed?"
        }
    ]
)

chat(messages)

Message(id='msg_01XrVRqhHJRPCzj25jQp8Quw', container=None, content=[TextBlock(citations=[CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n", document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationPageLocation(cited_text='[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n', document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text')], model='claude-sonnet-4-5-20250929', ro

In [5]:
article_text = '''
Earth is the third planet from the Sun and the only
astronomical object known to harbor life. This is
enabled by Earth being an ocean world, the only one in
the Solar System sustaining liquid surface water. Almost
all of Earth's water is contained in its global ocean,
covering 70.8% of Earth's crust. The remaining 29.2% of
Earth's crust is land, most of which is located in the
form of continental landmasses within Earth's land
hemisphere. Most of Earth's land is at least somewhat
humid and covered by vegetation, while large sheets of
ice at Earth's polar deserts retain more water than
Earth's groundwater, lakes, rivers, and atmospheric
water combined. Earth's crust consists of slowly moving
tectonic plates, which interact to produce mountain
ranges, volcanoes, and earthquakes. Earth has a liquid
outer core that generates a magnetosphere capable of
deflecting most of the destructive solar winds and
cosmic radiation.
'''

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "text",
                "media_type": "text/plain",
                "data": article_text
            },
            "title": "Earth Article",
            "citations": { "enabled": True}
        },
        {
            "type": "text",
            "text": "What is the Earth's position relative to the Sun?"
        }
    ]
)

chat(messages)

Message(id='msg_01KLrsbHvSQyucgERNtTxzZ3', container=None, content=[TextBlock(citations=[CitationCharLocation(cited_text='\nEarth is the third planet from the Sun and the only\nastronomical object known to harbor life. ', document_index=0, document_title='Earth Article', end_char_index=95, file_id=None, start_char_index=0, type='char_location')], text='Earth is the third planet from the Sun.', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=879, output_tokens=28, server_tool_use=None, service_tier='standard'), stop_details=None)